In [1]:
!pip install -q -U \
  transformers \
  datasets \
  accelerate \
  peft \
  trl \
  bitsandbytes \
  sentencepiece \
  protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 164.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.0 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4

In [2]:
from google.colab import files

uploaded = files.upload()

DATA_PATH = None
for filename in uploaded.keys():
    if filename.endswith(".jsonl"):
        DATA_PATH = filename
        break

assert DATA_PATH is not None, "Please upload fine_tune_chat.jsonl"

DATA_PATH

Saving fine_tune_chat.jsonl to fine_tune_chat.jsonl


'fine_tune_chat.jsonl'

In [2]:
from __future__ import annotations

import json
import os
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer


BASE_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Alternatives:
# BASE_MODEL = "Qwen/Qwen3-1.7B"
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

PROJECT_DIR = Path("/content/airplane_translation_finetune")
OUTPUT_DIR = PROJECT_DIR / "outputs"
ADAPTER_DIR = PROJECT_DIR / "adapter"
MERGED_DIR = PROJECT_DIR / "merged_model"

for path in [PROJECT_DIR, OUTPUT_DIR, ADAPTER_DIR, MERGED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [8]:

DATA_PATH = "./fine_tune_chat.jsonl"
dataset = load_dataset(
    "json",
    data_files=str(DATA_PATH),
    split="train",
)

dataset[0]

Generating train split: 0 examples [00:00, ? examples/s]

{'messages': [{'role': 'system',
   'content': "You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations.\n\nYour task is to translate the user's sentence between Turkish and English.\n\nRules:\n- If the input is Turkish, translate it into natural English.\n- If the input is English, translate it into natural Turkish.\n- Preserve the meaning, politeness level, urgency, and speaker intent.\n- Use simple, clear, practical language suitable for airplane passengers and cabin crew.\n- Do not add explanations.\n- Do not answer the user's request.\n- Do not roleplay.\n- Only return the translated sentence.\n- For emergency sentences, keep the translation direct and accurate.\n- For polite requests, preserve politeness naturally.\n- For announcements or crew instructions, use clear formal language.\n"},
  {'role': 'user',
   'content': 'Translate to Turkish: My bag does not fit in the overhead bin.'},
  {'role': 

In [9]:
split_dataset = dataset.train_test_split(
    test_size=0.02,
    seed=42,
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['messages', 'metadata'],
    num_rows: 49250
})
Dataset({
    features: ['messages', 'metadata'],
    num_rows: 1006
})


In [10]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HuggingFac-Write')

In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Pad token: <|eot_id|>
EOS token: <|eot_id|>


In [12]:
def format_chat_example(example):
    messages = example["messages"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


train_dataset = train_dataset.map(
    format_chat_example,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    format_chat_example,
    remove_columns=eval_dataset.column_names,
)

print(train_dataset[0]["text"])

Map:   0%|          | 0/49250 [00:00<?, ? examples/s]

Map:   0%|          | 0/1006 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 31 May 2026

You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations.

Your task is to translate the user's sentence between Turkish and English.

Rules:
- If the input is Turkish, translate it into natural English.
- If the input is English, translate it into natural Turkish.
- Preserve the meaning, politeness level, urgency, and speaker intent.
- Use simple, clear, practical language suitable for airplane passengers and cabin crew.
- Do not add explanations.
- Do not answer the user's request.
- Do not roleplay.
- Only return the translated sentence.
- For emergency sentences, keep the translation direct and accurate.
- For polite requests, preserve politeness naturally.
- For announcements or crew instructions, use clear formal language.<|eot_id|><|start_header_id|>user<|end_header_id|>

Tran

In [13]:
def token_length(example):
    return {
        "num_tokens": len(
            tokenizer(
                example["text"],
                add_special_tokens=False,
            )["input_ids"]
        )
    }


length_sample = train_dataset.select(range(min(2000, len(train_dataset)))).map(token_length)

lengths = length_sample["num_tokens"]

print("Min:", min(lengths))
print("Max:", max(lengths))
print("Mean:", sum(lengths) / len(lengths))
print("95th percentile:", sorted(lengths)[int(len(lengths) * 0.95)])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Min: 208
Max: 246
Mean: 221.7295
95th percentile: 231


In [14]:
MAX_SEQ_LENGTH = 256

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [15]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [17]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),

    max_length=MAX_SEQ_LENGTH,
    packing=False,

    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    logging_steps=25,
    save_steps=500,
    eval_steps=500,
    eval_strategy="steps",
    save_strategy="steps",

    save_total_limit=3,

    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8,

    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    report_to="none",
    dataset_text_field="text",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [19]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/49250 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/49250 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1006 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1006 [00:00<?, ? examples/s]

In [20]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,0.167659,0.170605,0.195588,1781751.000000,0.956688
1000,0.148206,0.153291,0.173300,3563140.000000,0.959159
1500,0.136521,0.141236,0.152960,5345391.000000,0.961168
2000,0.126307,0.132078,0.143204,7127912.000000,0.962854
2500,0.125424,0.127230,0.142774,8909982.000000,0.964040
3000,0.128821,0.126714,0.141362,10691831.000000,0.964102
3079,0.123875,0.126690,0.141331,10970666.000000,0.964173


TrainOutput(global_step=3079, training_loss=0.16951720642555684, metrics={'train_runtime': 4197.5085, 'train_samples_per_second': 11.733, 'train_steps_per_second': 0.734, 'total_flos': 6.726674701143245e+16, 'train_loss': 0.16951720642555684, 'epoch': 1.0})

In [21]:
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("Saved LoRA adapter to:", ADAPTER_DIR)

Saved LoRA adapter to: /content/airplane_translation_finetune/adapter


In [22]:
def translate(text: str, max_new_tokens: int = 80):
    messages = [
        {
            "role": "system",
            "content": "You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations. Only return the translated sentence.",
        },
        {
            "role": "user",
            "content": text,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [23]:
tests = [
    "Translate to Turkish: Can I have some water, please?",
    "Translate to English: Koltuğum nerede?",
    "Translate to Turkish: I cannot breathe.",
    "Translate to English: Çocuğum için su alabilir miyim?",
    "Translate to Turkish: Please fasten your seatbelt.",
]

for t in tests:
    print("INPUT:", t)
    print("OUTPUT:", translate(t))
    print("-" * 80)

INPUT: Translate to Turkish: Can I have some water, please?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


OUTPUT: Biraz su alabilir miyim, lütfen?
--------------------------------------------------------------------------------
INPUT: Translate to English: Koltuğum nerede?
OUTPUT: Where is my seat?
--------------------------------------------------------------------------------
INPUT: Translate to Turkish: I cannot breathe.
OUTPUT: Hâlâ nefes alamıyorum.
--------------------------------------------------------------------------------
INPUT: Translate to English: Çocuğum için su alabilir miyim?
OUTPUT: Can I get some water for my child?
--------------------------------------------------------------------------------
INPUT: Translate to Turkish: Please fasten your seatbelt.
OUTPUT: Lütfen emniyet kemerinizi bağlayın.
--------------------------------------------------------------------------------


In [26]:
!pip uninstall -y torchao

import gc
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

torch.cuda.empty_cache()
gc.collect()

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


1527

In [27]:


merge_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float16
)

base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=merge_dtype,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(
    base_model_for_merge,
    str(ADAPTER_DIR),
)

merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(
    str(MERGED_DIR),
    safe_serialization=True,
)

tokenizer.save_pretrained(str(MERGED_DIR))

print("Merged model saved to:", MERGED_DIR)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/airplane_translation_finetune/merged_model


In [28]:
merged_tokenizer = AutoTokenizer.from_pretrained(
    str(MERGED_DIR),
    use_fast=True,
    trust_remote_code=True,
)

merged_model = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16,
    trust_remote_code=True,
)

if merged_tokenizer.pad_token is None:
    merged_tokenizer.pad_token = merged_tokenizer.eos_token

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [29]:
def translate_merged(text: str, max_new_tokens: int = 80):
    messages = [
        {
            "role": "system",
            "content": "You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations. Only return the translated sentence.",
        },
        {
            "role": "user",
            "content": text,
        },
    ]

    prompt = merged_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = merged_tokenizer(prompt, return_tensors="pt").to(merged_model.device)

    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=merged_tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return merged_tokenizer.decode(generated, skip_special_tokens=True).strip()

In [30]:
for t in tests:
    print("INPUT:", t)
    print("OUTPUT:", translate_merged(t))
    print("-" * 80)

INPUT: Translate to Turkish: Can I have some water, please?
OUTPUT: Biraz su alabilir miyim, lütfen?
--------------------------------------------------------------------------------
INPUT: Translate to English: Koltuğum nerede?
OUTPUT: Where is my seat?
--------------------------------------------------------------------------------
INPUT: Translate to Turkish: I cannot breathe.
OUTPUT: Hava alamıyorum.
--------------------------------------------------------------------------------
INPUT: Translate to English: Çocuğum için su alabilir miyim?
OUTPUT: Can I get some water for my child?
--------------------------------------------------------------------------------
INPUT: Translate to Turkish: Please fasten your seatbelt.
OUTPUT: Lütfen emniyet kemerinizi bağlayın.
--------------------------------------------------------------------------------


In [31]:
import shutil
from google.colab import files

ADAPTER_ZIP = "/content/airplane_translation_adapter.zip"
MERGED_ZIP = "/content/airplane_translation_merged_model.zip"

shutil.make_archive(
    ADAPTER_ZIP.replace(".zip", ""),
    "zip",
    root_dir=str(ADAPTER_DIR),
)

shutil.make_archive(
    MERGED_ZIP.replace(".zip", ""),
    "zip",
    root_dir=str(MERGED_DIR),
)

files.download(ADAPTER_ZIP)
files.download(MERGED_ZIP)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
from google.colab import drive
from pathlib import Path
import shutil
import os

drive.mount("/content/drive")

DRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/MIS48B+/airplane_translation_model")
DRIVE_ADAPTER_DIR = DRIVE_EXPORT_DIR / "adapter_lora"
DRIVE_MERGED_DIR = DRIVE_EXPORT_DIR / "merged_final_model"

DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_ADAPTER_DIR.exists():
    shutil.rmtree(DRIVE_ADAPTER_DIR)

if DRIVE_MERGED_DIR.exists():
    shutil.rmtree(DRIVE_MERGED_DIR)

shutil.copytree(
    str(ADAPTER_DIR),
    str(DRIVE_ADAPTER_DIR),
)

shutil.copytree(
    str(MERGED_DIR),
    str(DRIVE_MERGED_DIR),
)

print("Saved adapter to:")
print(DRIVE_ADAPTER_DIR)

print("\nSaved merged final model to:")
print(DRIVE_MERGED_DIR)

Mounted at /content/drive
Saved adapter to:
/content/drive/MyDrive/MIS48B+/airplane_translation_model/adapter_lora

Saved merged final model to:
/content/drive/MyDrive/MIS48B+/airplane_translation_model/merged_final_model
